# Testing the RAW attention-score capture on PaliGemma

This notebook loads the **real PaliGemma** VLM and runs every correctness test
against it, proving that what the code captures is the genuine **pre-softmax**
text→vision attention scores `S = Q Kᵀ / √d` (NOT the softmax distribution).

It checks the three groups of cases:
* **(A) Structure** — image/text token counts partition the sequence; the sliced
  text→vision map has exactly `n_text × n_image` elements.
* **(B) Raw-vs-softmax discriminators** — the captured tensor has negatives, its
  rows don't sum to 1, and it isn't the softmax map → it is raw logits.
* **(C) Ground-truth identity** — `softmax(raw)` reproduces the model's own
  post-softmax attention (`argmax` match + numeric reconstruction), and
  `raw = log(post) + C` per row.

> **Runtime:** use a GPU runtime (`Runtime → Change runtime type → GPU`). PaliGemma
> is **gated** — accept its license at
> https://huggingface.co/google/paligemma-3b-pt-224 with the account you log in with.

## 1. Install dependencies

In [ ]:
!pip -q install -U transformers huggingface_hub safetensors pillow pytest

## 2. Clone the repo

In [ ]:
!git clone -q https://github.com/shubhamOjha1000/text_vision_attention_map.git
%cd text_vision_attention_map

## 3. Authenticate with HuggingFace
PaliGemma is gated. Accept the license once (link above), then run this cell and
paste a token (create one at https://huggingface.co/settings/tokens).

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 4. Build the PaliGemma probe
One forward pass captures BOTH the raw pre-softmax scores
(`GemmaAttention._raw_attn_scores`) and the model's own post-softmax attention
(`output[1]`), for every decoder layer. First run downloads the weights.

In [ ]:
import importlib.util, os

# Load our test module by explicit file path. Importing `tests.test_raw_attention`
# can fail in Colab when another top-level `tests` package on the Python path
# shadows this repo's tests/ directory. Loading by path avoids that entirely.
_path = os.path.join(os.getcwd(), "tests", "test_raw_attention.py")
assert os.path.exists(_path), f"not found: {_path} (did the clone + %cd cells run?)"
_spec = importlib.util.spec_from_file_location("test_raw_attention", _path)
T = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(T)

o = T.make_paligemma_output()
assert o is not None, "PaliGemma probe failed to load — check GPU / HF auth / license."
print("Probe:", o.name)
print("Sequence length L      :", o.L)
print("Decoder layers captured:", len(o.raw_scores), "->", sorted(o.raw_scores)[:5], "...")
print("raw_scores[0] shape    :", tuple(o.raw_scores[0].shape), "(H, L, L)")

## 5. Run ALL test cases on the real PaliGemma probe
Every invariant test, executed directly on the model we just probed.

In [ ]:
test_fns = [
    T.test_token_counts_partition_the_sequence,
    T.test_submatrix_element_count,
    T.test_raw_scores_contain_negative_values,
    T.test_raw_rows_do_not_sum_to_one,
    T.test_raw_differs_from_post_softmax,
    T.test_argmax_of_raw_matches_argmax_of_post,
    T.test_softmax_of_raw_reconstructs_post_softmax,
    T.test_within_row_log_difference_is_constant,
]

passed = 0
for fn in test_fns:
    try:
        fn(o)
        print(f"PASS  {fn.__name__}")
        passed += 1
    except AssertionError as e:
        print(f"FAIL  {fn.__name__}: {e}")

print(f"\n{passed}/{len(test_fns)} tests passed on PaliGemma.")

## 6. (A) Structure — concrete numbers
Token counts and the sub-matrix element count.

In [ ]:
n_img = int(o.image_token_mask.sum())
n_txt = int(o.text_token_mask.sum())
print("image tokens :", n_img, " (model-advertised:", o.expected_num_image_tokens, ")")
print("text  tokens :", n_txt)
print("image & text disjoint:", not bool((o.image_token_mask & o.text_token_mask).any()))
print("n_img + n_txt <= L    :", n_img + n_txt, "<=", o.L)

P = T.slice_text_vision(o.raw_scores[0], o.text_token_mask, o.image_token_mask).mean(0)
print("\nlayer-0 text->vision submatrix shape:", tuple(P.shape), "(L_t, L_v)")
print("elements:", P.numel(), "==  n_txt * n_img =", n_txt * n_img)

## 7. (B) Raw-vs-softmax discriminators — concrete numbers
Evidence the captured tensor is raw logits, not a softmax distribution.

In [ ]:
raw0 = o.raw_scores[0]
post0 = o.post_softmax[0]
print("min raw value           :", round(raw0.min().item(), 4), " (< 0  => softmax impossible)")
print("raw row sums (first 5)  :", [round(v, 3) for v in raw0.sum(-1).flatten()[:5].tolist()],
      " (softmax rows would be 1.0)")
print("post-softmax row sum     :", round(post0.sum(-1).flatten()[0].item(), 4), " (~1.0, as expected)")
import torch
print("raw == post-softmax?     :", torch.allclose(raw0, post0, atol=1e-4), " (must be False)")

## 8. (C) Ground-truth identity — softmax(raw) == model's attention
The definitive proof the raw capture is the true pre-softmax of the real attention.

In [ ]:
import torch
raw0 = o.raw_scores[0].double()
post0 = o.post_softmax[0].double()

recon = torch.softmax(raw0, dim=-1)
max_diff = (recon - post0).abs().max().item()
argmax_match = (raw0.argmax(-1) == post0.argmax(-1)).float().mean().item()

print("max |softmax(raw) - post_softmax| :", f"{max_diff:.2e}", " (~0  => exact reconstruction)")
print("argmax(raw) == argmax(post) frac  :", f"{argmax_match:.4f}", " (1.0 => same top key everywhere)")

# raw = log(post) + C : the per-row additive constant (softmax is shift-invariant)
trust = post0[0, 0] > 1e-4
c = (raw0[0, 0] - torch.log(post0[0, 0].clamp_min(1e-12)))[trust]
print("\nhead0 row0: raw - log(post) spread :", f"{(c.max() - c.min()).item():.2e}",
      " (~0 => constant C, i.e. raw ARE the logits behind post)")

## 9. (Optional) Full pytest suite
Runs the canonical suite from the command line — the synthetic probe **and** the
PaliGemma probe (this reloads the model, so it is slower). Expect all
`[synthetic]` and `[paligemma]` cases to pass.

In [ ]:
!python -m pytest tests/test_raw_attention.py -v